# Bonus B7 solution, notebook 1: turn the limerick row green

The worked answer to the challenge of `b7a_meet_langsmith.ipynb`. The setup repeats the notebook's helper, dataset
and evaluator, so this file runs on its own. It needs the LangSmith key, and its last cell deletes what it stored.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

# The LangSmith key is taken the same way
os.environ.setdefault("LANGSMITH_API_KEY", local.get("LANGSMITH_API_KEY") or "")
assert os.environ["LANGSMITH_API_KEY"], "Put LANGSMITH_API_KEY in .env at the repository folder (this lab needs a LangSmith account)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes | LangSmith key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes | LangSmith key found: yes


The helper of the notebook, with its tool and its instructions unchanged.

In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent


def split_bill(total: float, people: int, tip_percent: float) -> float:
    """Split a restaurant bill: add the tip as a percentage, divide by the number of people, round to cents."""
    return round(total * (1 + tip_percent / 100) / people, 2)


llm = ChatOpenAI(model=MODEL, reasoning_effort=EFFORT, timeout=60, max_retries=2)

INSTRUCTIONS = "You help friends split restaurant bills. Keep the reply to one short sentence."

helper = create_agent(llm, [split_bill], system_prompt=INSTRUCTIONS, name="bill_helper")

## Task 1: a second helper that is told what to do with other requests

One sentence is added to the instructions. It says what to do, and it gives the helper words to use, so the
reply is easy to check.

In [3]:
INSTRUCTIONS_2 = (
    "You help friends split restaurant bills. "
    "If a request is not about splitting a bill, do not do it: say that you only split bills. "
    "Keep the reply to one short sentence."
)

helper_2 = create_agent(llm, [split_bill], system_prompt=INSTRUCTIONS_2, name="bill_helper_2")

## Task 2: the same dataset, the same evaluator, two experiments

The client, the project name, tracing on, and the three examples as a dataset, as in the notebook.

In [4]:
import re
import uuid
import langsmith as ls
from langsmith import Client

client = Client()
PROJECT = "ai-agents-b7-" + uuid.uuid4().hex[:6]
ls.configure(enabled=True, project_name=PROJECT)

examples = [
    {"inputs": {"question": "Split 84.50 between 4 with a 15% tip."},
     "outputs": {"expect": r"24\.29"}},
    {"inputs": {"question": "Split 84.50 between 4 and add a 15 dollar tip."},
     "outputs": {"expect": r"24\.88"}},
    {"inputs": {"question": "Write me a limerick about tipping."},
     "outputs": {"expect": r"only|cannot|can't|can’t"}},
]

DATASET = PROJECT + "-questions"
dataset = client.create_dataset(DATASET, description="Three questions for the bill helper")
client.create_examples(dataset_id=dataset.id, examples=examples)


def right_answer(outputs: dict, reference_outputs: dict) -> bool:
    return bool(re.search(reference_outputs["expect"], outputs["answer"]))


print("project:", PROJECT)

project: ai-agents-b7-8a7bc0


One target per helper. The two differ only in the agent they ask, so one function builds both: `target_for(agent)`
returns a target that asks that agent.

In [5]:
def target_for(agent):
    def target(inputs: dict) -> dict:
        result = agent.invoke({"messages": [{"role": "user", "content": inputs["question"]}]})
        return {"answer": result["messages"][-1].text}
    return target

`run_experiment` wraps the `evaluate` call of the notebook, with the same arguments, so that we can call it twice.
It catches the link `evaluate` prints, which holds the account's id.

In [6]:
import contextlib
import io
from langsmith import evaluate


def run_experiment(target):
    printed = io.StringIO()
    with contextlib.redirect_stdout(printed), contextlib.redirect_stderr(printed):
        return evaluate(target, data=DATASET, evaluators=[right_answer],
                        experiment_prefix=PROJECT, disable_evaluator_tracing=True)


first = run_experiment(target_for(helper))
second = run_experiment(target_for(helper_2))

print("first experiment: ", first.experiment_name)
print("second experiment:", second.experiment_name)

first experiment:  ai-agents-b7-8a7bc0-26b3a2e8
second experiment: ai-agents-b7-8a7bc0-6e49e55d


## Task 3: the rows of both experiments

For each question we print the tick and the answer of the first experiment, then of the second.

In [7]:
def by_question(results):
    return {row["example"].inputs["question"]: row for row in results}


rows_1, rows_2 = by_question(first), by_question(second)

for example in examples:
    question = example["inputs"]["question"]
    print(question)
    for label, rows in [("first ", rows_1), ("second", rows_2)]:
        row = rows[question]
        score = row["evaluation_results"]["results"][0].score
        print("  ", label, "✅" if score else "❌", row["run"].outputs["answer"].replace("\n", " / "))

Split 84.50 between 4 with a 15% tip.
   first  ✅ With a 15% tip, each person pays $24.29.
   second ✅ With a 15% tip, it’s $24.29 per person.
Split 84.50 between 4 and add a 15 dollar tip.
   first  ❌ With the 15% tip, that’s $24.29 per person.
   second ✅ That’s $24.88 per person, including the $15 tip.
Write me a limerick about tipping.
   first  ❌ A diner who tipped with great flair   / Left a little extra to share;   / The server smiled bright,   / As they strolled through the night,   / And good cheer floated into the air.
   second ✅ I only split bills, not write limericks.


The limerick row turned green: the second helper answered with a one-line refusal that holds `only`, and called no
tool. The 15% row is green in both experiments.

The 15 dollar row needs a careful reading. In this run the first helper read 15 dollars as 15 percent and said
24.29, the wrong reply from Part A of notebook 1, and the second helper said 24.88. The new sentence is
about requests that are not about a bill, not about tips, so do not credit it with that row. The model writes a new reply on every run, and when we asked this question many
times the reply was wrong more often than right, so in your run either experiment may fail this row. One run of each experiment cannot
tell a fix from luck. Before you call a change better, run
both experiments more than once and compare the rows.

On the LangSmith page, the dataset lists both experiments. Select both and click **Compare** to see the rows side
by side: a row whose score went up is marked green, and one whose score went down red.

## Delete what this solution stored

The two experiments go with the dataset. The project received no runs of its own here, because every traced call
ran inside `evaluate`, so it is deleted only if it exists.

In [8]:
ls.configure(enabled=False)

client.delete_dataset(dataset_id=dataset.id)
client.delete_project(project_name=PROJECT) if client.has_project(PROJECT) else None

print("dataset gone:    ", not client.has_dataset(dataset_name=DATASET))
print("experiments gone:", not client.has_project(first.experiment_name) and not client.has_project(second.experiment_name))
print("project gone:    ", not client.has_project(PROJECT))

dataset gone:     True


experiments gone: True


project gone:     True


Three lines saying `True`: nothing from this solution is left in the account.

Back to [notebook 1](../b7a_meet_langsmith.ipynb) or [the lab's README](../README.md).